# LeapSpace Classification with Gold-Set Few-Shot Examples

This notebook uses the REVISED gold set (`data/leapspace_gold_set_revised.xlsx`), which fully supersedes the old 98-ticket gold set and its 14 Contact Reason / 56 Root Cause taxonomy. It:

1. Parses the new taxonomy from the `hierarchy` sheet (wide format -- one column per Contact Reason, Root Cause leaves listed top-to-bottom) and writes it into `definitions/leapspace_pipeline_definition.xlsx`'s `taxonomy` sheet
2. Rebuilds `contact_reason.json` / `generated/modules/contact_reason.py` via `rca build`
3. Loads the 117-ticket `tickets` sheet, deduplicates, resolves each (CR, RC) pair to a taxonomy leaf code (handling a known label alias), and converts it into `FeedbackStore` JSONL few-shot examples -- using synthetic sequential IDs, since the revised gold set has no ticket identifier column at all
4. Runs `rca run` on `leapspace_collapsed.csv` with those examples injected
5. Evaluates via 5-fold cross-validation

**Known accuracy tradeoff:** unlike the old gold set (which had an `Explanations` column) and unlike the Evolve product's Definition sheet, the `hierarchy` sheet has no descriptions -- only labels. Taxonomy L1/L2 entries here are label-only; classification quality depends more heavily on the few-shot examples than on taxonomy descriptions.

No ID matching is needed for classification -- gold examples are used as prompt context, not joined as rows to the bulk data.

**Note:** `scripts/build_leapspace_definition.py` and `scripts/run_kfold_eval.py` are now SUPERSEDED by this notebook (the taxonomy is parsed mechanically from the `hierarchy` sheet inline below, and k-fold is now inlined too, both mirroring `notebooks/05_evolve_classify_with_gold_set.ipynb`). Both scripts are left in place, unused, per this project's practice of disabling rather than deleting superseded code.

**Update -- LLM-checked corrections (`data/LeapSpace cr, rc llm checked.xlsx`).** Section 4 now overlays a round of LLM-checked label corrections on top of the revised gold set. That file is a review of the 91 tickets whose Root-Cause prediction disagreed with gold in a prior run of this notebook; its `CR final` / `RC Final` columns hold the reviewer's verdicts (36 filled, ~9 CR + ~9 RC labels flipped from the original gold). Corrections are merged by `uid`; rows the reviewer left blank keep their original gold label. The taxonomy is unchanged -- it is still built from the `hierarchy` sheet below, and already contains every label the corrections use.

## 0. Configuration -- edit this cell

In [ ]:
import os
from pathlib import Path as _P
_root = _P.cwd()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)  # the use-case folder

from pathlib import Path
BASE = Path().resolve()

# -- Paths --------------------------------------------------------------
PROJECT_DIR      = BASE
GOLD_SET_PATH    = BASE / "data/leapspace_gold_set_revised.xlsx"
CHECKED_SET_PATH = BASE / "data/LeapSpace cr, rc llm checked.xlsx"
DEFINITION_XLSX  = BASE / "definitions/leapspace_pipeline_definition.xlsx"
INPUT_CSV        = BASE / "data/leapspace_collapsed.csv"
OUTPUT_PATH      = BASE / "runs/output/leapspace_enriched.xlsx"
FEEDBACK_DIR     = BASE / "feedback/leapspace"
TAXONOMY_DIR     = BASE / "data/leapspace_taxonomies"
CODE_MAP_PATH    = BASE / "data/leapspace_code_map.json"

# -- Classification settings ------------------------------------------------
TEXT_COL          = "Conversation"
MODULES           = ["contact_reason"]
FEEDBACK_STRATEGY = "smart"
FEEDBACK_MAX      = 5
BATCH_SIZE        = 10
CONCURRENCY       = 4

# -- Subprocess environment ------------------------------------------------
# rca build / rca run print Unicode box-drawing separators.
# On Windows, a piped (non-console) stdout defaults to the system codepage
# (cp1252) unless UTF-8 is forced, which crashes those prints. Force it for
# every subprocess call below.
_SUBPROCESS_ENV = os.environ.copy()
_SUBPROCESS_ENV["PYTHONUTF8"] = "1"
_SUBPROCESS_ENV["PYTHONIOENCODING"] = "utf-8"

# -- Sanity check ------------------------------------------------------------
assert GOLD_SET_PATH.exists(), f"Gold set not found: {GOLD_SET_PATH}"
assert CHECKED_SET_PATH.exists(), f"Checked set not found: {CHECKED_SET_PATH}"
assert INPUT_CSV.exists(), f"Input CSV not found: {INPUT_CSV}"
assert DEFINITION_XLSX.exists(), f"Definition file not found: {DEFINITION_XLSX}"
print("Paths OK")

## 1. Build the taxonomy from the `hierarchy` sheet

The `hierarchy` sheet is a WIDE-format codebook -- one column per Contact Reason, with Root Cause leaves listed top-to-bottom in that column (unlike Evolve's long-format Definition sheet, which had one row per label). No descriptions exist anywhere in this sheet.

In [ ]:
import pandas as pd
import shutil

_REAL_CR_COLUMNS = [
    "Institutional user cannot access",
    "User not able to access",
    "Subscription cancellation and (or) refund",
    "Internal staff access",
    "Billing",
    "General info ",   # trailing space in the source column header -- stripped below
    "Not enough info",
    "Subscription request (incl. resub and trial)",
]
# 3 more columns ("Unnamed: 8/9/10") are entirely empty spacer columns -- dropped by
# simply not being in the list above. 4 more ("Unnamed: 11-14") contain unrelated
# email/org-entitlement scratch notes whose values never appear in the tickets
# sheet's RC column -- FLAGGED ASSUMPTION: excluded here as leftover notes, not
# confirmed with the SME who authored the sheet.

# Known parenthetical alias: the hierarchy sheet stores this as ONE cell, but the
# tickets sheet splits it into two separate (CR, RC) pairs under DIFFERENT parents
# -- see the label-resolution step below for the full cross-parent alias.
_HIERARCHY_CELL_ALIAS = {
    "Institution entitled incorrectly (Incomplete account setup)":
        "Institution entitled incorrectly",
}


def parse_hierarchy_sheet(hier_df: pd.DataFrame) -> list[dict]:
    """hier_df: the 'hierarchy' sheet, read with the default header row."""
    groups = []
    for col in _REAL_CR_COLUMNS:
        cr_label = col.strip()
        leaves_raw = hier_df[col].dropna().tolist()
        leaves = [_HIERARCHY_CELL_ALIAS.get(str(x).strip(), str(x).strip()) for x in leaves_raw]
        # One column ("User not able to access") lists the same leaf twice --
        # dedupe within-column, preserving order, so the taxonomy doesn't carry
        # a redundant duplicate entry.
        leaves = list(dict.fromkeys(leaves))
        groups.append({"cr_label": cr_label, "children": leaves})
    return groups


def build_taxonomy_rows(groups: list[dict]) -> pd.DataFrame:
    rows = []
    for g in groups:
        rows.append({
            "module_name": "contact_reason", "level": "L1",
            "label": g["cr_label"], "description": f"Contact reason: {g['cr_label']}", "notes": "",
        })
        for leaf in g["children"]:
            rows.append({
                "module_name": "contact_reason", "level": "L2",
                "label": leaf, "description": "", "notes": "",
            })
    return pd.DataFrame(rows)


hier_raw = pd.read_excel(GOLD_SET_PATH, sheet_name="hierarchy")
groups = parse_hierarchy_sheet(hier_raw)
print(f"Parsed {len(groups)} Contact Reason groups, {sum(len(g['children']) for g in groups)} Root Cause leaves")
for g in groups:
    print(f"  {g['cr_label']!r}  ({len(g['children'])} RC)")

In [ ]:
taxonomy_df = build_taxonomy_rows(groups)
print(f"Taxonomy sheet: {len(taxonomy_df)} rows "
      f"({(taxonomy_df['level']=='L1').sum()} L1, {(taxonomy_df['level']=='L2').sum()} L2)")
taxonomy_df.head(10)

Write the parsed taxonomy into the `taxonomy` sheet, preserving `modules`/`fields` sheets untouched. Unlike Evolve, LeapSpace's definition file already has just the single hierarchical `contact_reason` module with the right fields from the start -- no `modules`/`fields` edits are needed this time, only `taxonomy`.

In [ ]:
_def_bak = DEFINITION_XLSX.with_suffix(DEFINITION_XLSX.suffix + ".bak")
if not _def_bak.exists():
    shutil.copy2(DEFINITION_XLSX, _def_bak)
    print(f"Backed up: {_def_bak.name}")
else:
    print(f"Backup already exists, not overwriting: {_def_bak.name}")

with pd.ExcelWriter(DEFINITION_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    taxonomy_df.to_excel(writer, sheet_name="taxonomy", index=False)
print(f"Taxonomy sheet written: {len(taxonomy_df)} rows -> {DEFINITION_XLSX.name}")

In [ ]:
_check = pd.read_excel(DEFINITION_XLSX, sheet_name=None, engine="openpyxl")
print(f"modules: {len(_check['modules'])} row(s)")
print(f"fields: {len(_check['fields'])} row(s)")
print(f"taxonomy: {len(_check['taxonomy'])} rows "
      f"({(_check['taxonomy']['level']=='L1').sum()} L1, {(_check['taxonomy']['level']=='L2').sum()} L2)")

assert len(_check["taxonomy"]) == len(taxonomy_df)
assert (_check["taxonomy"]["module_name"] == "contact_reason").all()
assert len(_check["modules"]) == 1 and len(_check["fields"]) == 2
print("\nAll assertions passed.")

## 2. Run `rca build` to regenerate the taxonomy JSON and classifier module

Taxonomy JSON is written to a product-specific `--taxonomy-dir` (`data/leapspace_taxonomies/`), matching the existing convention already used by this notebook, to avoid overwriting `generated/taxonomies/` shared by other products.

In [ ]:
import subprocess, sys

TAXONOMY_DIR.mkdir(parents=True, exist_ok=True)

_old_tax_json = TAXONOMY_DIR / "contact_reason.json"
if _old_tax_json.exists():
    _tax_bak = _old_tax_json.with_name("contact_reason_OLD_14x56.json.bak")
    if not _tax_bak.exists():
        shutil.copy2(_old_tax_json, _tax_bak)
        print(f"Backed up: {_tax_bak.name}")

result = subprocess.run(
    [sys.executable, "-m", "rca_pipeline.cli", "build",
     "--input", str(DEFINITION_XLSX),
     "--taxonomy-dir", str(TAXONOMY_DIR),
     "--force", "contact_reason"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", cwd=str(BASE),
    env=_SUBPROCESS_ENV,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("rca build failed")
print("Build complete.")

In [ ]:
import json

with open(TAXONOMY_DIR / "contact_reason.json", encoding="utf-8") as f:
    tax_json = json.load(f)
n_l1 = len(tax_json)
n_l2 = sum(len(v.get("children", {})) for v in tax_json.values())
print(f"contact_reason.json: {n_l1} L1 groups, {n_l2} L2 leaves")
for code_, node in tax_json.items():
    print(f"  [{code_}] {node['label']}  ({len(node.get('children', {}))} children)")

assert n_l1 == 8 and n_l2 == 39
_module_py = BASE / "generated" / "modules" / "contact_reason.py"
assert _module_py.exists(), "contact_reason.py was not generated"
print(f"\ngenerated/modules/contact_reason.py regenerated at {_module_py.stat().st_mtime}")

## 3. Build the label -> code lookup (for `FeedbackStore` and evaluation)

In [ ]:
def build_label_to_code_lookup(taxonomy_json: dict) -> dict[str, dict[str, str]]:
    hierarchy = {}
    for l1_node in taxonomy_json.values():
        hierarchy[l1_node["label"]] = {
            leaf["label"]: code_ for code_, leaf in l1_node.get("children", {}).items()
        }
    return hierarchy


hierarchy = build_label_to_code_lookup(tax_json)

if CODE_MAP_PATH.exists():
    _map_bak = CODE_MAP_PATH.with_name("leapspace_code_map_OLD_14x56.json.bak")
    if not _map_bak.exists():
        shutil.copy2(CODE_MAP_PATH, _map_bak)
        print(f"Backed up: {_map_bak.name}")

with open(CODE_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump({"hierarchy": hierarchy}, f, indent=2, ensure_ascii=False)
print(f"Code map: {sum(len(v) for v in hierarchy.values())} CR->RC leaf codes -> {CODE_MAP_PATH.name}")

## 4. Load the gold set, deduplicate, and resolve labels to codes

The revised gold set has no unique ticket identifier column at all. Conversation text is 99.1% unique (1 exact duplicate pair); a synthetic sequential ID is assigned after deduplication. Per the Evolve precedent, this is sufficient -- gold examples are used purely as few-shot prompt context, never joined by ID to the raw classification data.

In [ ]:
gold = pd.read_excel(GOLD_SET_PATH, sheet_name="tickets")
gold["CR"] = gold["CR"].astype(str).str.strip()
gold["RC"] = gold["RC"].astype(str).str.strip()

_n_before = len(gold)
gold = gold.drop_duplicates(subset=["Conversation"], keep="first").reset_index(drop=True)
if len(gold) < _n_before:
    print(f"Dropped {_n_before - len(gold)} duplicate Conversation row(s) (identical labels on both copies)")

gold["uid"] = [f"gold_{i}" for i in range(len(gold))]

print(f"Gold set: {len(gold)} tickets (post-dedup)")
print(f"Columns: {gold.columns.tolist()}")
print("\nContact Reason (CR) distribution:")
print(gold["CR"].value_counts().to_string())
print("\nRoot Cause (RC) distribution:")
print(gold["RC"].value_counts().to_string())

### Overlay the LLM-checked corrections

`CHECKED_SET_PATH` reviews the 91 tickets whose Root-Cause prediction disagreed with gold in a prior run. Its `CR final` / `RC Final` columns hold the reviewer's authoritative verdicts (filled for 36 of 91 rows; ~9 Contact Reason and ~9 Root Cause labels differ from the original gold and are flipped here). Corrections are merged onto the base gold set by `uid`; the rows the reviewer left blank keep their original gold label. One label (`SD AI -> LeapSpace migration gap`) is repaired for a double-encoded arrow. Any (CR, RC) pair that still doesn't match the taxonomy after merging is handled by the existing resolution step below (excluded, not silently dropped).

In [ ]:
# --- Overlay the LLM-checked label corrections ----------------------------
import re

checked = pd.read_excel(CHECKED_SET_PATH, sheet_name="Sheet1")

# One label -- "SD AI -> LeapSpace migration gap" -- is mojibake in this file:
# the arrow U+2192 was double-encoded as the three chars U+00E2 U+2020 U+2019.
# Build both from chr() (keeps this source pure-ASCII) and repair that one
# known corruption so the label matches the taxonomy.
_MOJIBAKE_ARROW = chr(0x00E2) + chr(0x2020) + chr(0x2019)
_ARROW = chr(0x2192)
def _fix_arrow_mojibake(x):
    return x if pd.isna(x) else str(x).replace(_MOJIBAKE_ARROW, _ARROW)

for _c in ["CR final", "RC Final"]:
    checked[_c] = checked[_c].map(_fix_arrow_mojibake)
checked["uid"] = checked["uid"].astype(str).str.strip()

# Defense-in-depth: the base gold `uid` is re-derived fresh each run (synthetic
# gold_N ids by post-dedup row order). Confirm every checked uid still maps to
# the same conversation before trusting the join -- abort loudly if it drifted.
_norm_conv = lambda s: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", str(s).lower())).strip()
_base_conv = dict(zip(gold["uid"], gold["Conversation"].map(_norm_conv)))
_chk_conv  = dict(zip(checked["uid"], checked["conversation"].map(_norm_conv)))
for _u in checked["uid"]:
    assert _u in _base_conv and _base_conv[_u] == _chk_conv[_u], (
        f"checked uid {_u!r} does not map to the same base gold conversation -- "
        "the synthetic-uid derivation drifted; do not trust the merge"
    )
print(f"uid/conversation validation passed for all {len(checked)} checked rows")

# Overlay each label independently, only where the reviewer filled a verdict.
_final_cr = dict(zip(checked["uid"], checked["CR final"]))
_final_rc = dict(zip(checked["uid"], checked["RC Final"]))
_n_cr = _n_rc = 0
for _i, _row in gold.iterrows():
    _fc, _fr = _final_cr.get(_row["uid"]), _final_rc.get(_row["uid"])
    if pd.notna(_fc) and str(_fc).strip() and str(_fc).strip() != _row["CR"]:
        gold.at[_i, "CR"] = str(_fc).strip(); _n_cr += 1
    if pd.notna(_fr) and str(_fr).strip() and str(_fr).strip() != _row["RC"]:
        gold.at[_i, "RC"] = str(_fr).strip(); _n_rc += 1

print(f"Applied LLM-checked corrections: {_n_cr} CR labels, {_n_rc} RC labels changed")
print("\nCorrected Contact Reason (CR) distribution:")
print(gold["CR"].value_counts().to_string())
print("\nCorrected Root Cause (RC) distribution:")
print(gold["RC"].value_counts().to_string())

In [ ]:
gold.to_csv("data/leapspace_gold_set_revised_uid.csv", index=False)

**Label resolution.** The `tickets` sheet's (CR, RC) pairs mostly match the `hierarchy` sheet's parsed taxonomy, but one family of tickets needs an explicit alias: the hierarchy sheet stores `'Institution entitled incorrectly (Incomplete account setup)'` as ONE cell under CR=`'Institutional user cannot access'`, while the tickets sheet has these as TWO SEPARATE (CR, RC) pairs -- one under the SAME parent, one under a DIFFERENT parent (`'User not able to access'`). Both must resolve to the same taxonomy leaf; the alias is scoped at the (CR, RC) pair level, not just the RC label, since it reassigns that one ticket's effective L1 group. RC labels are also not globally unique across CR parents (e.g. 'Other', 'Not enough info' repeat), so matching resolves CR first, then looks up RC scoped to that parent.

In [ ]:
import re

_ALIAS_CR: dict[str, str] = {}

# Cross-parent (CR, RC) alias -- see markdown above for why this is scoped at the
# pair level rather than a plain RC-label alias.
_ALIAS_CR_RC_PAIR: dict[tuple[str, str], tuple[str, str]] = {
    ("user not able to access", "incomplete account setup"):
        ("institutional user cannot access", "institution entitled incorrectly"),
}


def _norm_label(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())


def resolve_cr_rc_to_code(cr_raw: str, rc_raw: str, hierarchy: dict) -> tuple:
    """Returns (leaf_code_or_None, matched_cr_label, matched_rc_label).
    RC matching is scoped to the resolved CR's children only -- RC labels are
    not globally unique."""
    cr_norm = _ALIAS_CR.get(_norm_label(cr_raw), _norm_label(cr_raw))
    rc_norm = _norm_label(rc_raw)

    if (cr_norm, rc_norm) in _ALIAS_CR_RC_PAIR:
        cr_norm, rc_norm = _ALIAS_CR_RC_PAIR[(cr_norm, rc_norm)]

    cr_lookup = {_norm_label(k): k for k in hierarchy}
    matched_cr = cr_lookup.get(cr_norm)
    if matched_cr is None:
        return None, "", ""

    rc_candidates = hierarchy[matched_cr]
    rc_lookup = {_norm_label(k): k for k in rc_candidates}
    matched_rc = rc_lookup.get(rc_norm)
    if matched_rc is None:
        return None, matched_cr, ""

    return rc_candidates[matched_rc], matched_cr, matched_rc


resolved = gold.apply(
    lambda row: resolve_cr_rc_to_code(row["CR"], row["RC"], hierarchy),
    axis=1, result_type="expand"
)
resolved.columns = ["leaf_code", "matched_cr", "matched_rc"]
gold = pd.concat([gold, resolved], axis=1)

n_resolved   = gold["leaf_code"].notna().sum()
n_unresolved = gold["leaf_code"].isna().sum()
print(f"Resolved: {n_resolved}/{len(gold)}  |  Unresolved: {n_unresolved}/{len(gold)}")

if n_unresolved:
    print("\nUnresolved rows (genuine gold-set labeling inconsistency -- excluded below, not silently dropped):")
    print(gold[gold["leaf_code"].isna()][["uid", "CR", "RC"]].to_string(index=False))

In [ ]:
gold_usable = gold[gold["leaf_code"].notna()].reset_index(drop=True)
print(f"Usable gold rows for few-shot/evaluation: {len(gold_usable)} / {len(gold)}")

**Optional, non-blocking:** attempt to recover a real `Reference #` for audit purposes by exact-matching gold Conversations against the bulk dataset (reusing the `norm()` normalization from `scripts/apply_gold_review.py`, the established precedent for this technique in this repo). Expect roughly 40% to match -- not reliable enough to depend on for anything functional, purely a nice-to-have.

In [ ]:
def norm(s: str) -> str:
    """Lowercase, strip all punctuation/unicode to spaces, collapse whitespace.
    (Copied from scripts/apply_gold_review.py's norm() for consistency.)"""
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", str(s).lower())).strip()


_bulk = pd.read_csv(INPUT_CSV)
_bulk_norm_to_ref: dict[str, list] = {}
for _, r in _bulk.iterrows():
    _bulk_norm_to_ref.setdefault(norm(r["Conversation"]), []).append(r["Reference #"])


def _match_reference(conv: str):
    refs = _bulk_norm_to_ref.get(norm(conv), [])
    return refs[0] if len(refs) == 1 else None


gold_usable["matched_reference"] = gold_usable["Conversation"].map(_match_reference)
_n_matched = gold_usable["matched_reference"].notna().sum()
print(f"Bonus audit: {_n_matched}/{len(gold_usable)} gold tickets exact-matched to a bulk "
      f"Reference # ({_n_matched/len(gold_usable)*100:.0f}%). Non-blocking -- not used for anything functional.")

## 5. Write the gold set to `FeedbackStore` JSONL

Each ticket becomes a feedback entry where `original == corrected` (the human label is treated as an authoritative correction). Unlike the old gold set (which had an `Explanations` column), the revised gold set has no free-text rationale column, so `reason` is left blank -- same as Evolve's own gold set.

In [ ]:
import json
from datetime import datetime, timezone

FEEDBACK_DIR.mkdir(parents=True, exist_ok=True)

_fb_path = FEEDBACK_DIR / "contact_reason_corrections.jsonl"
if _fb_path.exists():
    _fb_bak = FEEDBACK_DIR / "contact_reason_corrections_OLD_98.jsonl.bak"
    if not _fb_bak.exists():
        shutil.copy2(_fb_path, _fb_bak)
        print(f"Backed up: {_fb_bak.name}")


def write_feedback_jsonl(df: pd.DataFrame, out_path: Path) -> list[dict]:
    entries = []
    for _, row in df.iterrows():
        entries.append({
            "uid":       str(row["uid"]),
            "text":      str(row["Conversation"])[:1500],
            "original":  {"contact_reason_primary_code": row["leaf_code"]},
            "corrected": {"contact_reason_primary_code": row["leaf_code"]},
            "reason":    "",
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "module":    "contact_reason",
        })
    with open(out_path, "w", encoding="utf-8") as f:
        for e in entries:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")
    print(f"  {out_path.name}: {len(entries)} entries")
    return entries


cr_entries = write_feedback_jsonl(gold_usable, _fb_path)
print("\nFeedback JSONL ready.")

In [ ]:
print(json.dumps(cr_entries[0], indent=2, ensure_ascii=False))
sample_code = cr_entries[0]["corrected"]["contact_reason_primary_code"]
for cr, rc_map in hierarchy.items():
    for rc, code_ in rc_map.items():
        if code_ == sample_code:
            print(f"\n  {sample_code}  =>  Contact Reason: '{cr}'  /  Root Cause: '{rc}'")

## 6. Preview the full dataset

`leapspace_collapsed.csv` is unchanged -- 1,326 rows, classified in full below (no sampling), matching this notebook's existing default behavior.

In [ ]:
full_df = pd.read_csv(INPUT_CSV)
print(f"Full dataset: {len(full_df)} rows")
print(f"Columns: {full_df.columns.tolist()}")
full_df.head(3)

## 7. Run the pipeline

This calls `rca run` with `--feedback-dir` pointing to the JSONL just written, `--feedback-strategy smart` (adapts example selection per batch based on token-overlap with the batch content), and `--modules contact_reason`.

In [ ]:
import subprocess, sys, threading

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-m", "rca_pipeline.cli", "run",
    "--input",                str(INPUT_CSV),
    "--output",               str(OUTPUT_PATH),
    "--text-col",             TEXT_COL,
    "--modules",              "contact_reason",
    "--feedback-dir",         str(FEEDBACK_DIR),
    "--feedback-strategy",    FEEDBACK_STRATEGY,
    "--feedback-max-examples", str(FEEDBACK_MAX),
    "--batch-size",           str(BATCH_SIZE),
    "--concurrency",          str(CONCURRENCY),
    "--track-costs",
    "--taxonomy-dir",         str(TAXONOMY_DIR),
]

print("Running:", " ".join(str(c) for c in cmd))
print()

_err_lines: list[str] = []

def _capture_err(proc):
    for line in iter(proc.stderr.readline, ""):
        _err_lines.append(line)

proc = subprocess.Popen(
    cmd, text=True, encoding="utf-8", errors="replace",
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=1,
    env=_SUBPROCESS_ENV,
)
_t = threading.Thread(target=_capture_err, args=(proc,), daemon=True)
_t.start()
for _line in iter(proc.stdout.readline, ""):
    print(_line, end="", flush=True)
proc.wait()
_t.join(timeout=5)

if proc.returncode == 0:
    print(f"\n-> Pipeline complete. Output: {OUTPUT_PATH}")
else:
    print(f"\n-> Pipeline exited with code {proc.returncode}")
    if _err_lines:
        print("\n--- Errors (last 60 lines) ---")
        print("".join(_err_lines[-60:]))
    raise RuntimeError(f"rca run failed with exit code {proc.returncode}")

## 8. Inspect results

In [ ]:
results = pd.read_excel(OUTPUT_PATH)
print(f"Output rows: {len(results):,}")
print(f"Output columns: {results.columns.tolist()}")
results.head(5)

In [ ]:
print("Predicted Contact Reason (L1) distribution:")
print(results["contact_reason_class_l1_label"].value_counts().to_string())
print("\nPredicted Root Cause (L2) distribution:")
print(results["contact_reason_primary_label"].value_counts().to_string())

**Verification checks** -- codes match the taxonomy, and no (L1, L2) pair falls outside the valid hierarchy. **Note:** unlike Evolve, LeapSpace's `fields` sheet has no evidence/justification fields, so `contact_reason_evidence_grounded` will NOT appear in the output -- that is expected, not a bug (adding evidence-grounding fields is out of scope for this taxonomy-only update).

In [ ]:
_pattern_ok = results["contact_reason_primary_code"].astype(str).str.match(r"^CR[1-8]\.\d+$").all()
print("All codes match CR{1-8}.{n} pattern:", _pattern_ok)

def _check_pair(row):
    return row["contact_reason_primary_label"] in hierarchy.get(row["contact_reason_class_l1_label"], {})

_valid_pairs = results.apply(_check_pair, axis=1)
print(f"Valid (L1,L2) pairs: {_valid_pairs.sum()}/{len(results)}")
if not _valid_pairs.all():
    print(results.loc[~_valid_pairs, ["contact_reason_class_l1_label", "contact_reason_primary_label"]])

print("contact_reason_evidence_grounded present:", "contact_reason_evidence_grounded" in results.columns)
if "contact_reason_evidence_grounded" in results.columns:
    print("  value counts:", results["contact_reason_evidence_grounded"].value_counts(dropna=False).to_dict())

## 9. Gold-set evaluation -- 5-fold cross-validation

Uses all 115 usable gold tickets in rotation -- each fold holds out ~23 for evaluation while the other ~92 are injected as few-shot examples. Aggregating over 5 folds gives an unbiased accuracy estimate as if all 115 tickets were used as few-shot examples in production.

Inlined directly in this notebook (rather than calling `scripts/run_kfold_eval.py`, which hardcodes the old `Ticket Number`/`Contact Reason`/`Root Cause`/`Explanations` column names) -- mirrors `notebooks/05_evolve_classify_with_gold_set.ipynb`'s approach, avoiding a second, harder-to-sync copy of the label-resolution logic.

In [ ]:
import shutil, time
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold

N_FOLDS      = 5
RANDOM_STATE = 42
TMP_KFOLD_DIR      = BASE / "runs" / "tmp_kfold_leapspace"
KFOLD_RESULTS_PATH = BASE / "runs" / "output" / "leapspace_kfold_eval_results.xlsx"


def _write_fold_feedback(fold_train: pd.DataFrame, out_path: Path) -> None:
    entries = []
    for _, row in fold_train.iterrows():
        entries.append({
            "uid": str(row["uid"]),
            "text": str(row["Conversation"])[:1500],
            "original": {"contact_reason_primary_code": row["leaf_code"]},
            "corrected": {"contact_reason_primary_code": row["leaf_code"]},
            "reason": "",
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "module": "contact_reason",
        })
    with open(out_path, "w", encoding="utf-8") as f:
        for e in entries:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")


def _run_fold(fold_idx: int, test_df: pd.DataFrame, feedback_dir: Path):
    fold_dir = TMP_KFOLD_DIR / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    in_csv, out_xlsx = fold_dir / "fold_in.csv", fold_dir / "fold_out.xlsx"
    test_df[["uid", "conversation"]].to_csv(in_csv, index=False)
    cmd = [
        sys.executable, '-m', 'rca_pipeline.cli', 'run',
        "--input", str(in_csv), "--output", str(out_xlsx),
        "--text-col", "conversation", "--id-col", "uid",
        "--modules", "contact_reason",
        "--taxonomy-dir", str(TAXONOMY_DIR),
        "--feedback-dir", str(feedback_dir),
        "--feedback-strategy", FEEDBACK_STRATEGY,
        "--feedback-max-examples", str(FEEDBACK_MAX),
        "--batch-size", str(BATCH_SIZE), "--concurrency", str(CONCURRENCY),
    ]
    proc = subprocess.run(cmd, text=True, encoding="utf-8", errors="replace",
                           capture_output=True, env=_SUBPROCESS_ENV)
    if proc.returncode != 0 or not out_xlsx.exists():
        print(f"  [fold {fold_idx + 1}] failed: {proc.stderr[-2000:]}")
        return None
    return pd.read_excel(out_xlsx)


TMP_KFOLD_DIR.mkdir(parents=True, exist_ok=True)
cr_counts = gold_usable["CR"].value_counts()
stratify = gold_usable["CR"].where(
    cr_counts.reindex(gold_usable["CR"]).values >= N_FOLDS, "rare"
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
all_records = []
t0 = time.time()

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(gold_usable, stratify)):
    fold_train = gold_usable.iloc[train_idx].copy()
    fold_test  = gold_usable.iloc[test_idx].copy()
    print(f"Fold {fold_idx + 1}/{N_FOLDS}  ({len(fold_train)} few-shot | {len(fold_test)} test)")

    fb_dir = TMP_KFOLD_DIR / f"feedback_{fold_idx}"
    fb_dir.mkdir(parents=True, exist_ok=True)
    _write_fold_feedback(fold_train, fb_dir / "contact_reason_corrections.jsonl")

    test_input = fold_test.rename(columns={"Conversation": "conversation"})[["uid", "conversation"]].copy()
    test_input["uid"] = test_input["uid"].astype(str)

    t_fold = time.time()
    pred_df = _run_fold(fold_idx, test_input, fb_dir)
    fold_s = time.time() - t_fold
    if pred_df is None:
        print(f"  Fold {fold_idx + 1} failed -- skipping")
        continue

    pred_df["uid"] = pred_df["uid"].astype(str).str.strip()
    # Defense-in-depth: .set_index + .loc[uid] on a non-unique index returns a
    # DataFrame instead of a Series, silently corrupting downstream .get() calls
    # (this is exactly what happened with a duplicate gold-set ticket in the
    # Evolve notebook before this guard was added). Fold uids are expected
    # unique already (synthetic gold_N ids, deduplicated in Section 4 above).
    pred_df = pred_df.drop_duplicates(subset=["uid"], keep="first")
    assert pred_df["uid"].is_unique, "duplicate uid in fold output -- investigate before trusting .loc lookups"
    indexed = pred_df.set_index("uid")
    fold_rows = []
    for _, row in fold_test.iterrows():
        uid = str(row["uid"])
        if uid not in indexed.index:
            continue
        pred_row = indexed.loc[uid]
        cr_pred = str(pred_row.get("contact_reason_class_l1_label", "")).strip()
        rc_pred = str(pred_row.get("contact_reason_primary_label", "")).strip()
        fold_rows.append({
            "uid": uid, "gold_cr": row["matched_cr"], "gold_rc": row["matched_rc"],
            "pred_cr": cr_pred, "pred_rc": rc_pred,
        })
    all_records.extend(fold_rows)

    valid_cr = [r for r in fold_rows if r["pred_cr"]]
    cr_acc = accuracy_score([r["gold_cr"] for r in valid_cr], [r["pred_cr"] for r in valid_cr]) * 100 if valid_cr else float("nan")
    print(f"  Fold {fold_idx + 1} done in {fold_s:.0f}s  |  CR acc: {cr_acc:.1f}%  ({len(fold_rows)}/{len(fold_test)} matched)")

df_kfold = pd.DataFrame(all_records)
print(f"\nAggregate: {len(df_kfold)} predictions across {N_FOLDS} folds  ({(time.time()-t0)/60:.1f} min total)")

In [ ]:
for label, gold_col, pred_col in [("Contact Reason (L1)", "gold_cr", "pred_cr"), ("Root Cause (L2)", "gold_rc", "pred_rc")]:
    valid = df_kfold[df_kfold[pred_col].str.strip() != ""]
    if valid.empty:
        print(f"\n{label}: no predictions found")
        continue
    acc = accuracy_score(valid[gold_col], valid[pred_col]) * 100
    print(f"\n{label}  (n={len(valid)})  Accuracy: {acc:.1f}%")
    print(classification_report(valid[gold_col], valid[pred_col], zero_division=0))

df_kfold.to_excel(KFOLD_RESULTS_PATH, index=False)
shutil.rmtree(TMP_KFOLD_DIR, ignore_errors=True)
print(f"\nResults saved: {KFOLD_RESULTS_PATH}")

## 10. Record-level error analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

errors_cr = df_kfold[df_kfold["gold_cr"] != df_kfold["pred_cr"]]
print(f"Contact Reason errors: {len(errors_cr)}/{len(df_kfold)}")
if len(errors_cr):
    pairs = errors_cr.groupby(["gold_cr", "pred_cr"]).size().sort_values(ascending=False).head(15)
    print(pairs.to_string())

errors_rc = df_kfold[(df_kfold["gold_rc"] != df_kfold["pred_rc"]) & (df_kfold["pred_rc"].str.strip() != "")]
print(f"\nRoot Cause errors: {len(errors_rc)}/{len(df_kfold)}")
if len(errors_rc):
    pairs_rc = errors_rc.groupby(["gold_cr", "gold_rc", "pred_rc"]).size().sort_values(ascending=False).head(15)
    print(pairs_rc.to_string())

In [ ]:
errors_rc.to_csv("runs\output\leapspace_Root_Cause_errors.csv", index=False) 

In [ ]:
top_classes = df_kfold["gold_cr"].value_counts().head(8).index.tolist()
mask = df_kfold["gold_cr"].isin(top_classes)
cm = confusion_matrix(df_kfold.loc[mask, "gold_cr"], df_kfold.loc[mask, "pred_cr"], labels=top_classes)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=top_classes, yticklabels=top_classes, cmap="Blues", ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Gold (True)")
ax.set_title("Contact Reason confusion matrix (5-fold aggregate)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Adding more corrections to improve accuracy

After reviewing the output, add corrections for any misclassified tickets using the `FeedbackStore` directly. These will be picked up on the next run.

```python
from rca_pipeline.enhancements.feedback import FeedbackStore

store = FeedbackStore(FEEDBACK_DIR)

store.add_correction(
    module="contact_reason",
    uid="gold_42",
    text="transcript text here...",
    original={"contact_reason_primary_code": "CR1.2"},
    corrected={"contact_reason_primary_code": "CR3.1"},
    reason="The user's main concern was billing, not access itself",
)
```

The `reason` field has the highest weight in the `smart` strategy -- always fill it in.